# ViTextVQA — ablation đặc trưng OCR
Chọn kịch bản 1, 2, 3 hoặc 5 trong cell cấu hình. Kịch bản 4 là ABL1. Notebook dùng nhánh exp/vitextvqa-ocr-ablation, cấu hình ABL1 16×1, LR 1e-4, QA-ViT/Consformer/TWC tắt. Code này khác commit ABL1 lịch sử; cần chạy thêm kịch bản 4 trên cùng code để so sánh có kiểm soát.
Mỗi kịch bản có output và Hugging Face repo riêng. Chạy cell theo thứ tự.


## 1. Hugging Face token (để `finetune.py` tự push checkpoint mỗi epoch, tự resume khi Colab ngắt, và upload cuối)
Thêm secret `HF_TOKEN` (tab 🔑, bật *Notebook access*), **hoặc** chạy cell dưới rồi dán token.

In [ ]:
import os
try:
    from google.colab import userdata
    _t = userdata.get("HF_TOKEN")
    if _t:
        os.environ["HF_TOKEN"] = _t
except Exception as e:
    print("Colab Secrets:", e)
if not os.environ.get("HF_TOKEN"):
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("✅ Đã có HF_TOKEN từ Colab Secrets.")

## 2. Cấu hình — code (repo + commit) và module

In [ ]:
GIT_REPO       = "https://github.com/Kussssssss/ViSceT5.git"
GIT_COMMIT     = "exp/vitextvqa-ocr-ablation"
QAVIT          = "off"
OCR_SCENARIO = 1  # 1=text; 2=text+char; 3=text+char+bbox; 5=unaligned; 4=full control

# =========================================================================
# CHỌN 1 TRONG 4 CẤU HÌNH ABLATION (từ commit cc773b0: cả 4 cấu hình đều có đầy đủ Image + Question + OCR text + bbox + det + rec + char):
# 1. Baseline:                    USE_CONSFORMER = False, USE_TWC = False
# 2. Baseline + Consformer:       USE_CONSFORMER = True,  USE_TWC = False
# 3. Baseline + TWC:              USE_CONSFORMER = False, USE_TWC = True
# 4. Baseline + Consformer + TWC: USE_CONSFORMER = True,  USE_TWC = True (Full model)
# =========================================================================
USE_CONSFORMER = False
USE_TWC        = False
TWC_AUG_P      = 0.35
TWC_TAU        = 0.2
TWC_CONTRASTIVE = True    # True = Full · False = hai lượt + bản related, bỏ hoàn toàn TWC
LEARNING_RATE  = 1e-4   # phần pretrain sửa từ 3e-5 thành 1e-4 (new modules giữ 1e-4, scalar giữ 1e-3)
BATCH_SIZE     = 16
GRAD_ACCUM     = 1
DATASET        = "ViTextVQA"
PREDICT_CKPT   = "best"
PREDICT_MIN_DEV_EM = 0.0  # run đối chứng: 0.0 (điểm thấp cũng là kết quả cần có); run cải tiến: 31.0

import os
from huggingface_hub import get_token, HfApi
_tok = os.environ.get("HF_TOKEN") or get_token()
if _tok:
    os.environ["HF_TOKEN"] = _tok
HF_USER = "Kus669"
if _tok:
    try:
        HF_USER = HfApi(token=_tok).whoami().get("name") or HF_USER
    except Exception as e:
        print("whoami lỗi:", e)
else:
    print("⚠️ CHƯA có HF_TOKEN: checkpoint chỉ nằm trên Colab, mất khi ngắt kết nối!")

assert QAVIT in ("off", "no_question", "question"), "QAVIT: off | no_question | question"
assert 0 <= TWC_AUG_P <= 1 and TWC_TAU > 0 and BATCH_SIZE >= 1 and GRAD_ACCUM >= 1 and LEARNING_RATE > 0
RUN_NAME = "-".join([m for m, on in (("qavit", QAVIT == "question"), ("qavitnq", QAVIT == "no_question"),
                                     ("consformer", USE_CONSFORMER),
                                     (f"twc-p{int(round(100 * TWC_AUG_P))}-tau{TWC_TAU:g}" if TWC_CONTRASTIVE else f"twoviews-p{int(round(100 * TWC_AUG_P))}", USE_TWC)) if on] or ["base"])
assert OCR_SCENARIO in (1, 2, 3, 4, 5)
RUN_NAME += f"-ocr{OCR_SCENARIO}-bs{BATCH_SIZE}x{GRAD_ACCUM}"
print(f"Code: {GIT_REPO} @ {GIT_COMMIT}")
print(f"Cấu hình: QA-ViT={QAVIT} | Consformer={USE_CONSFORMER} | TWC={USE_TWC}"
      f"{f' (p={TWC_AUG_P}, τ={TWC_TAU}, TWC_CONTRASTIVE={TWC_CONTRASTIVE})' if USE_TWC else ''} | lr={LEARNING_RATE:g} | batch={BATCH_SIZE}x{GRAD_ACCUM} | RUN_NAME={RUN_NAME}")

## 3. Lấy code đúng commit và đặt biến môi trường
Clone `GIT_REPO` (lần đầu), chuyển sang `GIT_COMMIT`, chạy `setup.sh` của commit đó, rồi đặt biến môi trường theo những gì `run_pipeline.py` của commit đó đọc được. Chạy lại cell này mỗi khi đổi cell 2.

In [ ]:
import os, re, subprocess, yaml
REPO_DIR = "/content/ViSceT5"
if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone -q --branch {GIT_COMMIT} {GIT_REPO} {REPO_DIR}
%cd {REPO_DIR}
!git remote set-url origin {GIT_REPO}
!git fetch -q origin "+refs/heads/*:refs/remotes/origin/*" --tags


def _resolve(ref):
    for cand in (f"origin/{ref}", ref):
        out = subprocess.run(["git", "rev-parse", "--verify", "-q", cand + "^{commit}"], capture_output=True, text=True)
        if out.returncode == 0:
            return out.stdout.strip()
    raise ValueError(f"Không tìm thấy commit/nhánh/tag '{ref}' trong {GIT_REPO}")


CODE_SHA = _resolve(GIT_COMMIT)
CODE_TAG = CODE_SHA[:7]
subprocess.run(["git", "checkout", "-q", "--detach", CODE_SHA], check=True)
!git log --oneline -1
!bash setup.sh

READ = set(re.findall(r'"([A-Z][A-Z_]+)": "[a-z_]+"', open("run_pipeline.py", encoding="utf-8").read()))
YAML_CFG = yaml.safe_load(open("configs/finetune.yaml", encoding="utf-8"))
for _name in ("ABLATION_USE_IMAGE", "ABLATION_USE_OCR", "ABLATION_USE_OCR_AUG", "ABLATION_USE_QACLIP", "CLIP_LATE_ADAPTER",
              "DATASET_NAME", "GRADIENT_ACCUMULATION_STEPS", "HF_REPO", "LEARNING_RATE", "MOCK_TEST", "MODEL_NAME_OR_PATH",
              "NEW_MODULE_LR", "NUM_TRAIN_EPOCHS", "OCR_AUG_MODE", "OCR_FEATURE_NORM", "OCR_RELATED", "OCR_LATTICE_KEEP",
              "OUTPUT_DIR", "PER_DEVICE_EVAL_BATCH_SIZE", "PER_DEVICE_TRAIN_BATCH_SIZE", "QAVIT_MODE",
              "RESUME_CHECKPOINT_ID", "RESUME_FROM_CHECKPOINT", "SCALAR_LR", "STAGE", "TRAIN_CONFIG",
              "TWC_ADV_PROBABILITY", "TWC_ANSWER_WEIGHT", "TWC_CONTRASTIVE", "TWC_FEATURE", "TWC_LOSS", "TWC_MASK_SAME_TEXT", "TWC_OBJECTIVE",
              "TWC_QUEUE_SIZE", "TWC_QUEUE_START", "TWC_RELATED_TRAIN", "TWC_TEMPERATURE", "TWC_VIEW", "TWC_WEIGHT",
              "USE_TWC", "GIT_BRANCH", "GIT_COMMIT", "PREDICT_CKPT_DIR", "PREDICT_DATASET", "PREDICT_SPLIT"):
    os.environ.pop(_name, None)

_b = lambda v: "true" if v else "false"
OUTPUT_DIR = f"/content/ViSceT5/output/{CODE_TAG}_{DATASET}_{RUN_NAME}"
HF_REPO = f"{HF_USER}/ViSceT5-{CODE_TAG}-{DATASET.lower()}-{RUN_NAME}"
ENV = {"DATASET_NAME": DATASET, "ABLATION_USE_OCR": _b(USE_CONSFORMER), "USE_TWC": _b(USE_TWC),
       "TWC_ADV_PROBABILITY": str(TWC_AUG_P), "TWC_TEMPERATURE": str(TWC_TAU)}
if USE_TWC and not TWC_CONTRASTIVE:
    ENV["TWC_CONTRASTIVE"] = "false"
_missing = [k for k in ENV if k not in READ]
assert not _missing, f"Commit {CODE_TAG}: run_pipeline.py không đọc {_missing} → không áp dụng được cấu hình ở cell 2."
if "QAVIT_MODE" in READ:
    ENV["QAVIT_MODE"] = QAVIT
else:
    assert QAVIT != "no_question", f"Commit {CODE_TAG} chưa có QA-ViT no_question: đặt QAVIT = \"off\" hoặc \"question\"."
if "ABLATION_USE_QACLIP" in READ:
    ENV["ABLATION_USE_QACLIP"] = _b(QAVIT == "question")
if "ABLATION_USE_OCR_AUG" in READ:
    ENV["ABLATION_USE_OCR_AUG"] = _b(USE_TWC)
if "LEARNING_RATE" in READ and "LEARNING_RATE" in globals() and LEARNING_RATE is not None:
    ENV["LEARNING_RATE"] = str(LEARNING_RATE)
for _name, _key, _value in (("PER_DEVICE_TRAIN_BATCH_SIZE", "per_device_train_batch_size", BATCH_SIZE),
                            ("GRADIENT_ACCUMULATION_STEPS", "gradient_accumulation_steps", GRAD_ACCUM)):
    if _name in READ:
        ENV[_name] = str(_value)
    else:
        assert YAML_CFG.get(_key) == _value, (f"Commit {CODE_TAG} không đọc {_name}: batch lấy từ yaml "
                                              f"({_key} = {YAML_CFG.get(_key)}) → đặt BATCH_SIZE / GRAD_ACCUM đúng giá trị đó.")
ENV.update({
    "OCR_USE_CHAR": _b(OCR_SCENARIO != 1),
    "OCR_USE_BBOX": _b(OCR_SCENARIO in (3, 4, 5)),
    "OCR_USE_VISUAL": _b(OCR_SCENARIO in (4, 5)),
    "OCR_ALIGN_SPATIAL_VISUAL": _b(OCR_SCENARIO != 5),
})
assert all(k in READ for k in ENV), "Commit does not expose every requested flag"
os.environ.update(ENV, OUTPUT_DIR=OUTPUT_DIR, HF_REPO=HF_REPO, PYTHONUNBUFFERED="1")
print(f"Code {CODE_TAG} | biến môi trường: {ENV}")
print(f"OUTPUT_DIR={OUTPUT_DIR}\nHF_REPO={HF_REPO}\nPREDICT_CKPT={PREDICT_CKPT}")

## 4. Kiểm tra trước khi train
Cell đầu chạy unit test và dừng nếu có test lỗi. Cell sau là mock test `--smoke_test`: 8 mẫu train, 4 mẫu val, 3 bước; không push, không resume từ HF.


In [ ]:
!git checkout -q --detach {CODE_SHA}
!pip install -q pytest
_tests = subprocess.run(["python", "-m", "pytest", "tests/test_vqa_pipeline.py", "-q", "--tb=short"], capture_output=True, text=True)
print(_tests.stdout[-6000:], _tests.stderr[-1000:])
assert _tests.returncode == 0, "Có unit test lỗi: gửi output ở trên trước khi train."


In [ ]:
!git checkout -q --detach {CODE_SHA}
!HF_REPO= OUTPUT_DIR={OUTPUT_DIR}_mock MOCK_TEST=true STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5. Finetune (từ đầu, không pretrain) — log train có `twc_loss`, `twc_acc`, `twc_tau`, `view_loss` khi bật TWC

In [ ]:
!git checkout -q --detach {CODE_SHA}
!MOCK_TEST=false STAGE=finetune python3 run_pipeline.py 2>&1 | tee train_execution.log

## 5b. (Khi cần) Đẩy checkpoint local lên HF rồi train tiếp
Dùng khi đã train mà **chưa có `HF_TOKEN`** (checkpoint chỉ nằm trên Colab): chạy cell 1 (token) → cell 2 (cấu hình, đúng commit và công tắc của lần train) → cell 3 → cell này. Cell đẩy các `checkpoint-N` trong `OUTPUT_DIR` (kèm optimizer/scheduler để resume) lên `HF_REPO`, sau đó chạy lại **cell 5**: `finetune.py` tự resume từ checkpoint mới nhất trên HF và từ đó push mỗi epoch. Chỉ checkpoint đã lưu cuối epoch mới có; phần đang train dở của epoch sau bị mất. Nếu runtime Colab đã reset thì không còn gì để đẩy.

In [ ]:
!ls -d {OUTPUT_DIR}/checkpoint-* 2>/dev/null
!python scripts/hf_io.py upload_checkpoints --repo {HF_REPO} --dir {OUTPUT_DIR}

## 6. Dự đoán dev + test (chỉ khi dev EM ở epoch 4 hoặc 5 đạt `PREDICT_MIN_DEV_EM` ở cell 2) (`STAGE=predict` của `run_pipeline.py`) và chấm điểm
`predict.py` in EM/F1 trên dev và ghi file nộp cho test. File được chép vào `OUTPUT_DIR` để không đè giữa các cấu hình.
Test được chấm bằng `training/benchmark_eval.py` — **đúng giao thức benchmark** (cùng cách chấm các dòng baseline/LMM trong bài): EM, F1 token/char, BLEU-1..4/avg, CIDEr, BERTScore (mBERT).
Đáp án test lấy từ `eval_gt/<DATASET>_test_gt.json` trong repo (đã có `ViTextVQA`); dataset chưa có file thì cell hiện nút upload **ngay đầu cell** (trước khi predict).
Nếu `OUTPUT_DIR` (hoặc `predictions/` trên HF) đã có `submission_test.csv` thì cell **bỏ qua predict** và chỉ chấm điểm.
File nộp, log predict (có EM/F1 dev) và kết quả chấm test được upload vào thư mục `predictions/` của repo HF của cấu hình.

**Chọn checkpoint:** `PREDICT_CKPT` ở cell 2. Cell in bảng các checkpoint kèm EM/F1/CIDEr trên val của từng epoch rồi dùng checkpoint được chọn:
`best` = model tốt nhất (thư mục gốc, do `trainer.save_model` lưu sau `load_best_model_at_end`), `latest`, `epoch3`, hoặc `checkpoint-17580`.
Chỉ tải file inference của checkpoint đó (không tải optimizer). Kết quả của `best` nằm ở `OUTPUT_DIR` và `predictions/` trên HF;
của checkpoint khác nằm ở `OUTPUT_DIR/predict_checkpoint-N` và `predictions/checkpoint-N/` → không đè nhau.

**Predict lại sau khi Colab đã tắt:** chạy cell 1, 2, 3 (đặt đúng commit, công tắc của lần train và `PREDICT_CKPT`) rồi chạy thẳng cell này, bỏ qua mock và finetune.

In [ ]:
import os, shutil, glob, json
import sys, site, subprocess, glob
def _ensure_env():
    """transformers / huggingface_hub / evaluate / bert_score phải khớp nhau (cài dở sau setup.sh → lỗi import khi chấm BERTScore)."""
    probe = ("import huggingface_hub, transformers, evaluate, bert_score, pycocoevalcap;"
             "from transformers.models.auto.modeling_auto import MODEL_MAPPING; list(MODEL_MAPPING.keys())")
    if subprocess.run([sys.executable, "-c", probe], capture_output=True).returncode == 0:
        return
    print("⚠️ Thư viện chưa khớp (transformers / huggingface_hub / evaluate) → cài lại đúng bản của requirements.txt")
    for root in site.getsitepackages() + [site.getusersitepackages()]:
        for d in glob.glob(os.path.join(root, "~*")):  # thư mục pip bỏ dở: ~ransformers ...
            shutil.rmtree(d, ignore_errors=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.45.2", "huggingface_hub==0.25.2",
                    "evaluate", "bert-score", "git+https://github.com/salaniz/pycocoevalcap"], check=False)

def _gt_ok(gt_path, pred_csv):
    """GT test phải là đáp án thật và đúng tập đã predict; nếu không thì không chấm (tránh điểm 0 vô nghĩa)."""
    import pandas as pd
    try:
        anns = json.load(open(gt_path, encoding="utf-8"))["annotations"]
        gt = {int(a["id"]): str(a["answers"][0]).strip().lower() for a in anns if a.get("answers")}
    except Exception as e:
        print("❌ GT test không đọc được:", e)
        return False
    distinct = set(gt.values())
    if "your answer" in distinct or (len(gt) >= 100 and len(distinct) <= 5):
        print(f"❌ GT test là đáp án giả ({len(distinct)} giá trị khác nhau) → không chấm.")
        return False
    ids = set(pd.read_csv(pred_csv, keep_default_na=False).iloc[:, 0].astype(int))
    if len(ids & set(gt)) < 0.99 * len(ids):
        print(f"❌ ID dự đoán ({len(ids)}) không khớp GT ({len(gt)}): chung {len(ids & set(gt))} → không chấm.")
        return False
    print(f"✅ GT test hợp lệ: {len(gt)} câu, {len(distinct)} đáp án khác nhau, khớp {len(ids & set(gt))}/{len(ids)} ID dự đoán.")
    return True
_ensure_env()
from huggingface_hub import HfApi, hf_hub_download
!git checkout -q --detach {CODE_SHA}
!git fetch -q origin exp/twc-bs4x2-ablation && git show FETCH_HEAD:scripts/hf_io.py > /content/hf_io_new.py
_states = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*/trainer_state.json"), key=lambda p: int(p.split("checkpoint-")[-1].split("/")[0]))
if not _states:
    _remote = sorted((f for f in HfApi().list_repo_files(HF_REPO) if f.endswith("/trainer_state.json")), key=lambda p: int(p.split("checkpoint-")[-1].split("/")[0]))
    _states = [hf_hub_download(HF_REPO, _remote[-1])] if _remote else []
_devs = [100 * h["eval_em"] for h in json.load(open(_states[-1]))["log_history"] if "eval_em" in h] if _states else []
print("Dev EM theo epoch:", [round(d, 2) for d in _devs])
if max(_devs[3:5], default=0) < PREDICT_MIN_DEV_EM:
    raise SystemExit(f"Dev EM ở epoch 4, 5 chưa đạt {PREDICT_MIN_DEV_EM}: bỏ qua predict.")
TEST_GT_JSON = f"/content/ViSceT5/eval_gt/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    TEST_GT_JSON = f"/content/{DATASET}_test_gt.json"
if not os.path.exists(TEST_GT_JSON):
    from google.colab import files
    print(f"⬆️ Chọn file {os.path.basename(TEST_GT_JSON)} để upload (bấm Cancel nếu chưa có → vẫn predict, chưa chấm test)")
    up = files.upload()
    if up:
        shutil.move(os.path.abspath(next(iter(up))), TEST_GT_JSON)
print("GT test:", TEST_GT_JSON if os.path.exists(TEST_GT_JSON) else "chưa có")

import json
tag = "" if DATASET == "ViTextVQA" else f"{DATASET}_"
# HF đã có file dự đoán của checkpoint này → không tải model 1.4GB, không predict lại: chỉ tải file dự đoán về, chấm và up điểm.
if os.path.exists("/content/pred_info.json"):
    os.remove("/content/pred_info.json")
!python /content/hf_io_new.py download_predictions --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/pred_info.json
PRED_INFO = json.load(open("/content/pred_info.json", encoding="utf-8"))
SCORE_ONLY = os.path.exists(os.path.join(PRED_INFO["pred_dir"], f"submission_{tag}test.csv"))
if SCORE_ONLY:
    print("✅ HF đã có file dự đoán → bỏ qua tải model và predict, chỉ chấm điểm rồi up kết quả.")
    MODEL_DIR, PRED_DIR, PRED_REPO_PATH = None, PRED_INFO["pred_dir"], PRED_INFO["pred_path_in_repo"]
else:
    if os.path.exists("/content/ckpt_info.json"):
        os.remove("/content/ckpt_info.json")
    !python scripts/hf_io.py prepare --repo {HF_REPO} --dir {OUTPUT_DIR} --ckpt {PREDICT_CKPT} --info_out /content/ckpt_info.json
    CKPT_INFO = json.load(open("/content/ckpt_info.json", encoding="utf-8"))
    MODEL_DIR, PRED_DIR, PRED_REPO_PATH = CKPT_INFO["model_dir"], CKPT_INFO["pred_dir"], CKPT_INFO["pred_path_in_repo"]
    assert os.path.exists(os.path.join(MODEL_DIR, "model.safetensors")), f"Không có model trong {MODEL_DIR}"

pred = os.path.join(PRED_DIR, f"submission_{tag}test.csv")
if os.path.exists(pred):
    print(f"✅ Đã có {pred} → bỏ qua predict, chỉ chấm điểm.")
else:
    for sp in ("dev", "test"):
        if os.path.exists(f"/content/ViSceT5/output/submission_{tag}{sp}.csv"):
            os.remove(f"/content/ViSceT5/output/submission_{tag}{sp}.csv")
    !MOCK_TEST=false STAGE=predict PREDICT_SPLIT=both PREDICT_DATASET={DATASET} PREDICT_CKPT_DIR={MODEL_DIR} HF_REPO= python3 run_pipeline.py 2>&1 | tee train_execution.log
    for sp in ("dev", "test"):
        src = f"/content/ViSceT5/output/submission_{tag}{sp}.csv"
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(PRED_DIR, os.path.basename(src)))
            print("→", os.path.join(PRED_DIR, os.path.basename(src)))
    shutil.copy2("/content/ViSceT5/train_execution.log", os.path.join(PRED_DIR, "predict_log.txt"))
log = os.path.join(PRED_DIR, "predict_log.txt")
dev_lines = [l.strip() for l in open(log, encoding="utf-8", errors="ignore") if "[dev] EM=" in l] if os.path.exists(log) else []
if dev_lines:
    for line in dev_lines[-2:]:
        print(line)
if os.path.exists(TEST_GT_JSON) and os.path.exists(pred) and _gt_ok(TEST_GT_JSON, pred):
    _ensure_env()
    !python -m training.benchmark_eval --gt {TEST_GT_JSON} --pred {pred} --out {PRED_DIR}/eval_test.json
else:
    print("ℹ️ Chưa đặt TEST_GT_JSON → chưa chấm test.")

if os.environ.get("HF_TOKEN") and HF_REPO:
    !python scripts/hf_io.py upload_predictions --repo {HF_REPO} --dir {PRED_DIR} --path_in_repo {PRED_REPO_PATH} --files submission_{tag}dev.csv submission_{tag}test.csv predict_log.txt eval_test.json
else:
    print("⚠️ Không có HF_TOKEN/HF_REPO → file dự đoán chỉ nằm trên Colab, hãy tải về trước khi ngắt phiên.")